In [ ]:
import ants
import os
import numpy as np
import nibabel as nib
from nibabel.processing import conform 

parent_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/'
roi_folder = os.path.join(parent_folder,'MMPS ROI')

# Set up folders, rois, maps for subject
subject_id = '2014'
subject_folder = os.path.join(parent_folder,'WCMyofascial' + subject_id)
roi_subfolder = os.path.join(roi_folder,subject_id)
maps_folder = os.path.join(subject_folder,'maps')
maps = os.listdir(maps_folder)
map_files = [s for s in maps if '.nii.gz' in s]
map_files = [s for s in map_files if '_reg' not in s]

# --- Downsampling ------
# Load MPRAGE
mprage_file = os.path.join(subject_folder,'nifti','1_SAG_3D_MPRAGE_GRAPPA3_MPR_cor_SAG_3D_MPRAGE_GRAPPA3_MPR_cor.nii.gz')
mprage_image = nib.load(mprage_file)
# Load segmentation
seg_file = os.path.join(roi_folder,subject_id,subject_id + '_mprage.nii')
mprage_seg = nib.load(seg_file)
dixon_array = mprage_image.get_fdata()
seg_array = mprage_seg.get_fdata()

# Downsample mprage and segmentation to 180 x 180 in plane
resamp_mprage = conform(mprage_image,out_shape=(180, 36, 180),voxel_size=(1.3333333, 2.0400033, 1.3333333),order=3)
resamp_mprage.to_filename(os.path.join(subject_folder,'mprage_resampled.nii.gz'))
resamp_seg = conform(mprage_seg,out_shape=(180, 36, 180),voxel_size=(1.3333333, 2.0400033, 1.3333333),order=0)
resamp_seg.to_filename(os.path.join(subject_folder,'mask_resampled.nii.gz'))

# ------ Registration -------
# Load upsampled mprage
fixed_ants = ants.image_read(os.path.join(subject_folder,'mprage_resampled.nii.gz'))
print(fixed_ants.shape)
nx,ny,nz = fixed_ants.shape 
mprage_nib = nib.load(os.path.join(subject_folder,'mprage_resampled.nii.gz'))
affine_fixed = mprage_nib.affine

# Load lowest diffusion time b0
b0_file = os.path.join(subject_folder,'derivatives/all/0022ms/dwiec.nii')
moving_ants4d = ants.image_read(b0_file)
print(moving_ants4d.shape)
moving_array = moving_ants4d.numpy()
b0_array = moving_array[...,0]

moving_b0 = ants.from_numpy(
    b0_array,
    spacing=moving_ants4d.spacing[:3],
    origin=moving_ants4d.origin[:3],
    direction=moving_ants4d.direction[:3,:3]
)
ants.image_write(moving_b0,os.path.join(subject_folder,'b0.nii.gz'))
reg = ants.registration(fixed=fixed_ants, moving=moving_b0, type_of_transform='Affine', smoothing_sigmas=[0,0,0], shrink_factors=[1,1,1]) # this is the main warp from dwi b0 to mprage

# Now iterate through diffusion times
diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']
for i in range(0,len(diffusion_times)):
    dt = diffusion_times[i]
    print(dt)

    # # --- register dwi images ----
    # dwi_path = os.path.join(subject_folder,'derivatives','all',dt,'dwiec.nii')
    # moving_ants4d = ants.image_read(dwi_path)
    # moving_array = moving_ants4d.numpy()
    # _,_,_,nvols = moving_ants4d.shape

    # # set up empty array
    # registered_im = np.zeros([nx,ny,nz,nvols])

    # for j in range(0,nvols): # Iterate through volumes
    #         vol_ants =  ants.from_numpy(moving_array[:,:,:,j],spacing=moving_ants4d.spacing[:3],origin=moving_ants4d.origin[:3],direction=moving_ants4d.direction[:3,:3])

    #         if i==0: # only need to do the dwi to dixon warp if lowest diffusion time b0 
    #             # print(files[i],'vol',j)
    #             # warped_moving_vol = ants.apply_transforms(fixed=fixed_ants, moving=vol_ants,transformlist=reg['fwdtransforms'],interpolator='bSpline')
    #             print('n')

    #         else: # all other volumes and files need both warps
    #             reg_vol2vol = ants.registration(fixed=moving_b0,moving=vol_ants,type_of_transform='Affine') # this is the registration between that volume and the lowest dt b0
    #             # warped_moving_vol = ants.apply_transforms(fixed=fixed_ants,moving=vol_ants,transformlist=[reg['fwdtransforms'][0], reg_vol2vol['fwdtransforms'][0]],interpolator='bSpline') # need to apply both transforms
    #             warped_moving_vol = ants.apply_transforms(fixed=fixed_ants,moving=vol_ants,transformlist=[reg_vol2vol['fwdtransforms'][0]],interpolator='bSpline')
    #         # ants.image_write(warped_moving_vol, 'test.nii.gz')
    #         warped_numpy = warped_moving_vol.numpy()
    #         registered_im[:,:,:,j] = warped_numpy

    #     # before, sep, after = files[i].partition(".nii")
    #     # new_name = before + '_reg' + sep + after
    # new_filepath = os.path.join(subject_folder,'derivatives','all',dt,'dwiec_reg.nii')
    # nib.save(nib.Nifti1Image(registered_im, affine_fixed),new_filepath)

    # ---- register maps -----
    # vol_b0 =  ants.from_numpy(moving_array[:,:,:,j],spacing=moving_ants4d.spacing[:3],origin=moving_ants4d.origin[:3],direction=moving_ants4d.direction[:3,:3])
    # reg_vol2vol = ants.registration(fixed=moving_b0,moving=vol_b0,type_of_transform='Affine') # this is the reg that needs to be applied to the maps

    # matched_maps = [s for s in map_files if dt in s]
    # print(matched_maps)

    # if i > 0:
    #     for map in matched_maps:
    #         map_filepath = os.path.join(maps_folder,map)
    #         map_ants = ants.image_read(map_filepath)
    #         warped_moving_vol = ants.apply_transforms(fixed=fixed_ants,moving=map_ants,transformlist=[reg_vol2vol['fwdtransforms'][0]],interpolator='bSpline')
    #         ants.image_write(warped_moving_vol, map_filepath[:-7] + '_reg.nii.gz')







(180, 36, 180)
(120, 120, 36, 34)
0022ms
0042ms
0081ms
0156ms
0300ms
